In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import anndata as ad
import crested
import keras
from scipy.stats import pearsonr, spearmanr

/users/rprest2/.conda/envs/Crested/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1776982003.966586   15798 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
#Load the genome
genome = crested.Genome(
        fasta="/scratch/rprest2/indices/mm10_encode.fa",
        chrom_sizes="/scratch/rprest2/indices/mm10_no_alt.chrom.sizes.tsv")
crested.register_genome(genome)

2026-04-23T22:07:17.650312+0000 INFO Genome mm10_encode registered.


In [7]:
adata = ad.read_h5ad("/scratch/rprest2/Enhancer-Creation/input/training_inputs/01_a_ft_training_set.h5ad")
adata_specific = ad.read_h5ad("/scratch/rprest2/Enhancer-Creation/input/training_inputs/02_finetune_DA_peaks.h5ad") 

BM_01TS_prmean_2114_nonorm_ep10 = crested.utils.load_model("/scratch/rprest2/Enhancer-Creation/input/training_models/BM_02TS_prmean_2114_nonorm/checkpoints/10.keras")
BM_01TS_prmean_2114_nonorm_ep10__FT_DA8414 = crested.utils.load_model("/scratch/rprest2/Enhancer-Creation/input/training_models/BM_02TS_prmean_2114_nonorm_ep10__FT_DA8414_LR1e-4/checkpoints/02.keras")
BM_01TS_prmean_2114__nonorm_ep10__FT_Gini1 = crested.utils.load_model("/scratch/rprest2/Enhancer-Creation/input/training_models/BM_02TS_prmean_2114_nonorm_ep10__FT_Gini-1_LR1e-4/checkpoints/04.keras")

#── Post-training evaluation ──────────────────────────────────────────────────

print("Evaluating model on held-out test chromosomes (chr9, chr18)...")
#Base model and FT model predictions on test set of adata_specific (cell type specific peaks)
#At some point, I should also compare base model and FT models on test set of adata (all peaks)

predictions_base = crested.tl.predict(adata_specific, BM_01TS_prmean_2114_nonorm_ep10)
adata_specific.layers["Base model"] = predictions_base.T  # adata expects (classes, genes) instead of (genes, classes)
predictions_ft_DA = crested.tl.predict(adata_specific, BM_01TS_prmean_2114__nonorm_ep10__FT_Gini1)
adata_specific.layers["Finetune on Gini=1"] = predictions_ft_DA.T
predictions_ft_Gi = crested.tl.predict(adata_specific, BM_01TS_prmean_2114_nonorm_ep10__FT_DA8414)
adata_specific.layers["Finetune on DA peaks"] = predictions_ft_Gi.T

Evaluating model on held-out test chromosomes (chr9, chr18)...


I0000 00:00:1776982174.006650   15798 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1776982174.256123   16352 service.cc:153] XLA service 0x7fa258035200 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776982174.256146   16352 service.cc:161]   StreamExecutor [0]: NVIDIA L4, Compute Capability 8.9 (Driver: 12.8.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.21.0)
I0000 00:00:1776982174.276584   16352 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776982174.367375   16352 cuda_dnn.cc:461] Loaded cuDNN version 92100
W0000 00:00:1776982177.249430   16671 hlo_rematerialization.cc:3204] Can't reduce memory use below 16.33GiB (17530392412 bytes) by rematerialization; only reduced to 18.97GiB (20370685984 bytes), down from 18.97GiB (20370685984 bytes) originally
W0000 00:00:1776982189.534266   16670 hlo_rematerializat

65/66 ━━━━━━━━━━━━━━━━━━━━ 0s 258ms/step

W0000 00:00:1776982262.187410   17242 hlo_rematerialization.cc:3204] Can't reduce memory use below 16.41GiB (17623270031 bytes) by rematerialization; only reduced to 508.31GiB (545794228256 bytes), down from 508.31GiB (545794228256 bytes) originally
W0000 00:00:1776982272.963145   16351 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 507.94GiB (rounded to 545396883968)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1776982272.963220   16351 bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
I0000 00:00:1776982272.963230   16351 bfc_allocator.cc:1056] Bin (256): 	Total Chunks: 75, Chunks in use: 75. 18.8KiB allocated for chunks. 18.8KiB in use in bin. 1.3KiB client-requested in use in bin.
I0000 00:00:1776982272.963243   16351 bfc_allocator.cc:1056] Bin (512): 	Total

66/66 ━━━━━━━━━━━━━━━━━━━━ 146s 1s/step
66/66 ━━━━━━━━━━━━━━━━━━━━ 20s 277ms/step
66/66 ━━━━━━━━━━━━━━━━━━━━ 20s 278ms/step


In [14]:
#Evaluate each model on the test set of adata_specific
crested.tl.evaluate(
    adata_specific,
    model='Base model',
    metrics=crested.tl.default_configs('peak_regression_mean')
)

crested.tl.evaluate(
    adata_specific,
    model='Finetune on Gini=1',
    metrics=crested.tl.default_configs('peak_regression_mean')
)

crested.tl.evaluate(
    adata_specific,
    model='Finetune on DA peaks',
    metrics=crested.tl.default_configs('peak_regression_mean')
)




2026-04-23T22:17:50.526559+0000 INFO Test CosineMSELogLoss: -0.3192
2026-04-23T22:17:50.527080+0000 INFO Test mean_absolute_error: 0.0031
2026-04-23T22:17:50.527461+0000 INFO Test mean_squared_error: 0.0000
2026-04-23T22:17:50.527934+0000 INFO Test cosine_similarity: 0.8371
2026-04-23T22:17:50.528438+0000 INFO Test pearson_correlation: 0.5538
2026-04-23T22:17:50.528910+0000 INFO Test concordance_correlation_coefficient: 0.4648
2026-04-23T22:17:50.529345+0000 INFO Test pearson_correlation_log: 0.6239
2026-04-23T22:17:50.831618+0000 INFO Test CosineMSELogLoss: -0.1136
2026-04-23T22:17:50.832180+0000 INFO Test mean_absolute_error: 0.0034
2026-04-23T22:17:50.832660+0000 INFO Test mean_squared_error: 0.0001
2026-04-23T22:17:50.833184+0000 INFO Test cosine_similarity: 0.8210
2026-04-23T22:17:50.834037+0000 INFO Test pearson_correlation: 0.4758
2026-04-23T22:17:50.834444+0000 INFO Test concordance_correlation_coefficient: 0.1869
2026-04-23T22:17:50.834909+0000 INFO Test pearson_correlation_lo

In [19]:
print(adata_specific.layers.keys())
print(adata_specific)
# Print the first 5 rows of peak metadata
print(adata_specific.var.head())
# Print the first 5 rows of sample metadata
print(adata_specific.obs.head())
# Look at the predicted heights for the first 5 peaks across the first 3 samples from the Base Model
base_predictions = adata_specific.layers['Base model']
print("Base Model Predictions (First 5 peaks, 3 samples):")
print(base_predictions[:5, :3])

KeysView(Layers with keys: Base model, Finetune on Gini=1, Finetune on DA peaks)
AnnData object with n_obs × n_vars = 16 × 8414
    obs: 'file_path'
    var: 'chr', 'start', 'end', 'split', 'da_class', 'fold', 'fdr'
    layers: 'Base model', 'Finetune on Gini=1', 'Finetune on DA peaks'
                       chr    start      end  split da_class      fold  \
region                                                                   
chr1:3044769-3046883  chr1  3044769  3046883  train  met_low  0.893567   
chr1:3045494-3047608  chr1  3045494  3047608  train  met_low  0.953468   
chr1:3131204-3133318  chr1  3131204  3133318  train  met_low  1.403196   
chr1:3163961-3166075  chr1  3163961  3166075  train  met_low  1.248629   
chr1:3201576-3203690  chr1  3201576  3203690  train  met_low  1.382720   

                           fdr  
region                          
chr1:3044769-3046883  0.047525  
chr1:3045494-3047608  0.035199  
chr1:3131204-3133318  0.014318  
chr1:3163961-3166075  0.00908

In [11]:
crested.pl.corr.violin(adata_specific)

In [12]:
crested.pl.corr.heatmap(
    adata_specific,
    split="test",
    log_transform=True,
    vmax=1,
    vmin=0,
)
plt.savefig("output/CREsted_Evaluation/02_Model_CellTypeHeatMap.png")
plt.close()

FileNotFoundError: [Errno 2] No such file or directory: 'output/CREsted_Evaluation/02_Model_CellTypeHeatMap.png'

In [ ]:
#This is the gene locus for NFKB1. Jesse says this is not the best gene locus to use. I should select 3 gene locuses for the Met-high and Met-low peaks to use for these plots.
crested.pl.region.bar(
    data = adata_specific,
    region= "chr3:135711468-135713582",
    pred_color= "lightblue",
    truth_color= "blue"
)
plt.savefig("output/CREsted_Evaluation/02_Region_Bar_Plot.png")

Additionally, I should answer the question: How do the models perform on predicting Met-High peaks in met-high samples and met-low peaks in met-low samples?